# E1 - procesar datos con pandas

Respuesta a las preguntas de autoevaluacion:

1. ¿Qué diferencia una Series de un DataFrame?
Una Series es una sola columna con su índice. Un DataFrame es una tabla con varias columnas, y cada columna es una Series.

2. ¿Por qué `pathlib` frente a rutas absolutas?
Una ruta absoluta (como `/Users/mi_usuario/...`) solo funciona en mi ordenador. Con `pathlib` y una ruta relativa (`Datos/ventas.csv`) el código funciona en cualquier ordenador, sea Mac o Windows.

3. ¿Qué imprime tu diagnóstico antes de limpiar?
El tamaño de la tabla (`shape`), el tipo de cada columna (`dtypes`), las primeras filas (`head`), los nulos por columna (`isna().sum()`) y resúmenes con `describe` y `value_counts`. Después anoto qué filas parecen inválidas y por qué.

4. ¿Por qué validar antes de `groupby`?
Porque si agrupo con datos malos (nulos, precios negativos, texto en columnas numéricas), los totales salen mal y parecen correctos sin serlo. Primero hay que dejar solo los datos válidos.

5. ¿Cuándo evitarías `apply(axis=1)`?
Cuando la operación se puede hacer con columnas enteras (por ejemplo `5 * df["a"] + 6 * df["b"]`), porque es más rápido y más claro. Solo lo usaría si no hay otra forma.

6. ¿Qué debe contener `calidad_datos.json` en el E1?
Cuatro datos: `filas_totales`, `filas_validas`, `filas_invalidas` e `importe_total`.

In [ ]:
## Parte 1 — Diagnóstico (8 min)

1. Carga el CSV con `pathlib.Path` (sin rutas absolutas hardcodeadas).
2. Imprime `shape`, `dtypes` y `isna().sum()`.
3. Lista por escrito (comentario o markdown) **qué filas parecen inválidas** y por qué.

In [ ]:
from datetime import date
from pathlib import Path
import json
import pandas as pd 
import numpy as np 

#1
DATA_DIR = Path("Datos")
ruta_ventas = DATA_DIR / ("ventas.csv")
ventas = pd.read_csv(ruta_ventas)

#2
print(f"Veamos como es el formato general del csv:\n {ventas.head()}\n")
print(f"tamaño del csv:\n {ventas.shape}\n")
print(f"diferentes tipos del csv:\n {ventas.dtypes}\n")
print(f"numero de nulos en el csv:\n {ventas.isna().sum()}\n")

#3 Que filas aparecen invalidas y porque:
#Esto lo he respondido despues de hacer el codigo de la siguiente celda

# Una fila es inválida si unidades o precio_unitario no son números mayores que 0.
# Hay 10 filas inválidas (número de fila de pandas):
# Fila 3: unidades es 0
# Fila 41: precio_unitario es 0
# Fila 54: unidades vacío
# Fila 63: unidades vacío
# Fila 66: precio_unitario vacío
# Fila 84: unidades tiene el texto "na" en vez de un número
# Fila 88: unidades es -2 (negativo)
# Fila 109: unidades es 0 y precio_unitario es -1.0
# Fila 134: precio_unitario es -5.0 (negativo)
# Fila 149: precio_unitario es -32.0 (negativo)

Veamos como es el formato general del csv:
         fecha region   producto unidades  precio_unitario cliente_id
0  2026-01-09    Sur  Cable Kit        7            12.50       C005
1  2026-03-14  Oeste    Gateway       25           120.00       C032
2  2026-01-26  Oeste    Hub IoT        1            88.96       C016
3  2026-02-05  Norte    Gateway        0           120.00       C010
4  2026-02-07  Norte    Hub IoT       15            85.00       C027

tamaño del csv:
 (150, 6)

diferentes tipos del csv:
 fecha                  str
region                 str
producto               str
unidades               str
precio_unitario    float64
cliente_id             str
dtype: object

numero de nulos en el csv:
 fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64



In [25]:
# Validacion para ver que filas son validas o no y responder a #3

"""
Reglas mínimas:

- `unidades` numérica y `> 0`
- `precio_unitario` numérico y `> 0`
- columnas derivadas: `importe = unidades * precio_unitario` solo en válidos
Nota:
lo que hace errors="coerce" es Lo que no se puede convertir, como "na", pasa a nulo (NaN) en lugar de dar un error.


Devuelve `(validos, errores)`.

**Checkpoint:** con el CSV del curso debes obtener **140 válidas** y **10 inválidas**.

"""

def validar_ventas(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    copia_datos = frame.copy() #lo usamos para trabajar sobre una copia
    copia_datos["unidades"] = pd.to_numeric(copia_datos["unidades"], errors = "coerce")
    copia_datos["precio_unitario"] = pd.to_numeric(copia_datos["precio_unitario"], errors = "coerce")
    
    # regla: unidades y precio existen y son mayores que 0
    ok = (
        copia_datos["unidades"].notna()
        & (copia_datos["unidades"] > 0)
        & copia_datos["precio_unitario"].notna()
        & (copia_datos["precio_unitario"] > 0)
    )

    validos = copia_datos.loc[ok].copy()
    errores = copia_datos.loc[~ok].copy()
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores

In [26]:
validos, errores = validar_ventas(ventas)
print("válidas:", len(validos), "| inválidas:", len(errores))
errores

válidas: 140 | inválidas: 10


,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0.0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4.0,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5.0,NaN,C025
84,2026-02-15,Este,Gateway,NaN,120.0,C022
88,2026-02-07,Este,Hub IoT,-2.0,85.0,C012
109,2026-02-19,Norte,Cable Kit,0.0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3.0,-5.0,C007
149,2026-02-13,Sur,Sensor B,8.0,-32.0,C020



## Parte 3 — Agregaciones (8 min)

Sobre `validos`:

1. Importe total por `region` (ordenado desc).
2. Top 3 `producto` por importe.
3. `cliente_id` con más de una compra.

In [ ]:
# 1 Importe total por región de mayor a menor
validos.groupby("region")["importe"].sum().sort_values(ascending=False)
#ascending=False significa "no ascendente", o sea, de más a menos.

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64

In [28]:
# 2 Top 3 productos por importe
validos.groupby("producto")["importe"].sum().sort_values(ascending=False).head(3)

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64

In [29]:
# 3. Clientes con más de una compra
compras = validos["cliente_id"].value_counts()
compras[compras > 1]

cliente_id
C027    9
C032    8
C004    8
C035    7
C013    6
C009    6
C005    5
C022    5
C038    5
C010    4
C018    4
C002    4
C040    4
C025    4
C015    4
C024    4
C023    4
C034    4
C037    4
C016    3
C003    3
C007    3
C011    3
C029    3
C033    3
C030    3
C028    2
C036    2
C021    2
C039    2
C031    2
C001    2
C020    2
C017    2
Name: count, dtype: int64

In [30]:
validos.to_csv(DATA_DIR / "ventas_limpias.csv", index=False)

calidad = {
    "filas_totales": int(len(ventas)),
    "filas_validas": int(len(validos)),
    "filas_invalidas": int(len(errores)),
    "importe_total": float(validos["importe"].sum()),
}
(DATA_DIR / "calidad_datos.json").write_text(json.dumps(calidad, indent=2, ensure_ascii=False))

113

In [31]:
print((DATA_DIR / "calidad_datos.json").read_text())

{
  "filas_totales": 150,
  "filas_validas": 140,
  "filas_invalidas": 10,
  "importe_total": 96598.17000000001
}
